# Single-Cell Gene Expression using VAEs — Complete Colab Notebook

**Research question:** How does the decoder distribution change reconstruction and learned representations of cells?

Five phases: **(1) Data & QC → (2) Gaussian VAE → (3) Poisson & Negative Binomial VAEs → (4) Evaluation → (5) Save results**.

Run cells **top to bottom**. Uses **PBMC 3k**, beginner-friendly PyTorch, no pretrained models or Lightning. A GPU is optional.

**Important:** The model learns a compressed **representation of each cell**, not the cell-type name. Numbers in the count matrix are *observed RNA-derived counts*, not exact molecules in the original cell.

## Setup

In [ ]:
!pip -q install "scanpy[skmisc]" umap-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import torch
import torch.nn as nn
from scipy import sparse
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, TensorDataset
import umap
from pathlib import Path

np.random.seed(42)
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
Path("results").mkdir(exist_ok=True)
print("Device:", device)

## Phase 1 — Load, inspect and prepare PBMC 3k

Matrix shape: **cells × genes**. Preserve **raw integer gene counts** for the Poisson/NB models. QC thresholds are starting values suitable for this introductory dataset, not universal rules.

In [ ]:
adata = sc.datasets.pbmc3k()
print("Original shape:", adata.shape)

adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, ["n_genes_by_counts", "total_counts", "pct_counts_mt"]):
    ax.hist(adata.obs[col], bins=40)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
adata = adata[
    (adata.obs["n_genes_by_counts"] >= 200)
    & (adata.obs["n_genes_by_counts"] < 2500)
    & (adata.obs["pct_counts_mt"] < 5)
].copy()

sc.pp.filter_genes(adata, min_cells=3)

indices = np.random.permutation(adata.n_obs)
n_train = int(0.70 * len(indices))
n_val = int(0.15 * len(indices))
train = adata[indices[:n_train]].copy()
val = adata[indices[n_train:n_train + n_val]].copy()
test = adata[indices[n_train + n_val:]].copy()

sc.pp.highly_variable_genes(train, n_top_genes=2000, flavor="seurat_v3")
genes = train.var_names[train.var["highly_variable"]]

def get_counts(data):
    x = data[:, genes].X
    return (x.toarray() if sparse.issparse(x) else np.asarray(x)).astype("float32")

X_train = get_counts(train)
X_val = get_counts(val)
X_test = get_counts(test)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Percentage of zeros:", round(100 * np.mean(X_train == 0), 2))
print("First cell's first ten counts:", X_train[0, :10])

### Two input types

- **Gaussian VAE:** normalize each cell *within the 2,000 selected genes* to 10,000 counts, then apply `log1p`. The output is continuous.
- **Poisson and NB VAEs:** encoder receives `log1p(raw counts)`, but decoder likelihood is computed on **raw integer counts**. Library size is the total count over the **selected genes**, matching decoder softmax proportions.

In [ ]:
def encoder_input(counts, model_name):
    if model_name == "gaussian":
        library = np.maximum(counts.sum(axis=1, keepdims=True), 1)
        return np.log1p(counts / library * 10000).astype("float32")
    return np.log1p(counts).astype("float32")

plt.figure(figsize=(10, 4))
plt.imshow(np.log1p(X_train[:80, :100]), aspect="auto", cmap="viridis")
plt.xlabel("Genes")
plt.ylabel("Cells")
plt.colorbar(label="log(1 + count)")
plt.title("Selected gene counts")
plt.show()

## Phase 2 — Gaussian VAE

Encoder: 2,000 → 256 → 128 → two 20-dimensional heads (**mean** and **log variance**). Decoder: 20 → 128 → 256 → 2,000.

We sample `z = mu + exp(0.5*logvar)*epsilon`. The Gaussian reconstruction objective uses **0.5 × sum of squared errors per cell** (unit observation variance).

In [ ]:
class VAE(nn.Module):
    def __init__(self, n_genes, latent_dim=20, model_name="gaussian"):
        super().__init__()
        self.model_name = model_name

        self.encoder = nn.Sequential(
            nn.Linear(n_genes, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )
        self.mean = nn.Linear(128, latent_dim)
        self.logvar = nn.Linear(128, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, n_genes)
        )
        if model_name == "nb":
            self.log_theta = nn.Parameter(torch.zeros(n_genes))

    def encode(self, x):
        hidden = self.encoder(x)
        return self.mean(hidden), self.logvar(hidden).clamp(-12, 12)

    def forward(self, x, library):
        mu, logvar = self.encode(x)
        epsilon = torch.randn_like(mu)
        z = mu + torch.exp(0.5 * logvar) * epsilon
        output = self.decoder(z)

        if self.model_name != "gaussian":
            proportions = torch.softmax(output, dim=1)
            output = (proportions * library[:, None]).clamp(min=1e-8)

        return output, mu, logvar

In [ ]:
def loss_function(model, x, prediction, mu, logvar):
    if model.model_name == "gaussian":
        recon = 0.5 * (prediction - x).square().sum(dim=1).mean()

    elif model.model_name == "poisson":
        nll = prediction - x * torch.log(prediction) + torch.lgamma(x + 1)
        recon = nll.sum(dim=1).mean()

    else:
        theta = torch.nn.functional.softplus(model.log_theta).clamp(min=1e-4)
        nll = (
            torch.lgamma(x + theta) * (-1)
            + torch.lgamma(theta) + torch.lgamma(x + 1)
            + theta * (torch.log(theta + prediction) - torch.log(theta))
            + x * (torch.log(theta + prediction) - torch.log(prediction))
        )
        recon = nll.sum(dim=1).mean()

    kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(dim=1).mean()
    return recon + kl, recon, kl

In [ ]:
def make_loader(counts, model_name, shuffle=False):
    x = torch.tensor(encoder_input(counts, model_name))
    target = x if model_name == "gaussian" else torch.tensor(counts)
    library = torch.tensor(counts.sum(axis=1).clip(min=1), dtype=torch.float32)
    return DataLoader(TensorDataset(x, target, library), batch_size=128, shuffle=shuffle)

def evaluate(model, counts, repeats=3):
    model.eval()
    sums = np.zeros(3)
    with torch.no_grad():
        for enc, target, library in make_loader(counts, model.model_name):
            enc, target, library = enc.to(device), target.to(device), library.to(device)
            batch_values = np.zeros(3)
            for _ in range(repeats):
                pred, mu, logvar = model(enc, library)
                values = loss_function(model, target, pred, mu, logvar)
                batch_values += np.array([v.item() for v in values]) / repeats
            sums += batch_values * len(enc)
    return sums / len(counts)

def train_model(model_name, epochs=100, latent_dim=20):
    torch.manual_seed(42)
    model = VAE(X_train.shape[1], latent_dim, model_name).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    best_loss = float("inf")
    best_state = None
    patience = 0
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        train_total = 0.0
        for enc, target, library in make_loader(X_train, model_name, shuffle=True):
            enc, target, library = enc.to(device), target.to(device), library.to(device)
            prediction, mu, logvar = model(enc, library)
            loss, _, _ = loss_function(model, target, prediction, mu, logvar)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_total += loss.item() * len(enc)

        train_loss = train_total / len(X_train)
        val_loss = evaluate(model, X_val)[0]
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"{model_name}: epoch {epoch+1}, train={train_loss:.2f}, val={val_loss:.2f}")
        if patience == 15:
            break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f"results/{model_name}_vae.pt")
    plt.figure(figsize=(6, 4))
    plt.plot(train_losses, label="train")
    plt.plot(val_losses, label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Negative ELBO (without constants)")
    plt.legend()
    plt.title(model_name.upper())
    plt.savefig(f"results/{model_name}_loss.png", bbox_inches="tight")
    plt.show()
    return model

In [ ]:
gaussian_vae = train_model("gaussian")
print("Gaussian test [loss, reconstruction, KL]:", evaluate(gaussian_vae, X_test, repeats=10))

## Phase 3 — Poisson and Negative Binomial VAEs

**Poisson:** mean = library size × decoder softmax proportions; variance = mean.

**NB:** same positive mean, plus **learned positive gene-specific dispersion** `theta`. Its variance is `mu + mu²/theta`. This accommodates overdispersion.

Both optimize **raw-count negative log likelihood + Gaussian latent KL**.

In [ ]:
poisson_vae = train_model("poisson")
print("Poisson test [loss, reconstruction, KL]:", evaluate(poisson_vae, X_test, repeats=10))

In [ ]:
nb_vae = train_model("nb")
print("NB test [loss, reconstruction, KL]:", evaluate(nb_vae, X_test, repeats=10))

## Phase 4 — Compare reconstruction and representations

**Important:** Gaussian reconstruction is computed on *normalized log expression*, whereas Poisson and NB reconstruction are computed on *raw counts*. Do **not** rank the Gaussian loss directly against count likelihood values.

We compare PCA and Gaussian VAE using test reconstruction MSE, and compare Poisson vs NB using the **same count observation scale**.

In [ ]:
models = {"Gaussian": gaussian_vae, "Poisson": poisson_vae, "NB": nb_vae}
results = []
for name, model in models.items():
    total, recon, kl = evaluate(model, X_test, repeats=10)
    results.append({"model": name, "test_loss": total, "reconstruction": recon, "kl": kl})

results = pd.DataFrame(results)
display(results)
results.to_csv("results/metrics.csv", index=False)

In [ ]:
def latent_means(model, counts):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(encoder_input(counts, model.model_name), device=device)
        mu, logvar = model.encode(x)
    return mu.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    z = latent_means(model, X_test)
    coords = umap.UMAP(random_state=42).fit_transform(z)
    ax.scatter(coords[:, 0], coords[:, 1], s=10)
    ax.set_title(name + " latent UMAP")
plt.tight_layout()
plt.savefig("results/umap.png", bbox_inches="tight")
plt.show()

In [ ]:
x_train_log = encoder_input(X_train, "gaussian")
x_test_log = encoder_input(X_test, "gaussian")
pca = PCA(n_components=20)
pca.fit(x_train_log)
pca_pred = pca.inverse_transform(pca.transform(x_test_log))
pca_mse = np.mean((pca_pred - x_test_log) ** 2)

with torch.no_grad():
    x = torch.tensor(x_test_log, device=device)
    mu, _ = gaussian_vae.encode(x)
    vae_pred = gaussian_vae.decoder(mu).cpu().numpy()

vae_mse = np.mean((vae_pred - x_test_log) ** 2)
print("PCA MSE:", pca_mse)
print("Gaussian VAE MSE:", vae_mse)

In [ ]:
def predicted_mean(model, counts):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(encoder_input(counts, model.model_name), device=device)
        library = torch.tensor(counts.sum(axis=1).clip(min=1), device=device)
        z, _ = model.encode(x)
        raw = model.decoder(z)
        mean = torch.softmax(raw, dim=1) * library[:, None]
    return mean.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
observed = X_test.mean(axis=0)
for ax, name, model in zip(axes, ["Poisson", "NB"], [poisson_vae, nb_vae]):
    pred = predicted_mean(model, X_test)
    ax.scatter(observed, pred.mean(axis=0), s=7, alpha=0.5)
    ax.set(xlabel="Observed mean count", ylabel="Predicted mean count", title=name)
plt.tight_layout()
plt.savefig("results/count_means.png", bbox_inches="tight")
plt.show()

In [ ]:
for name, model in [("Poisson", poisson_vae), ("NB", nb_vae)]:
    mu = predicted_mean(model, X_test).clip(min=1e-8)
    if name == "Poisson":
        p_zero = np.exp(-mu)
    else:
        theta = torch.nn.functional.softplus(model.log_theta).detach().cpu().numpy()
        p_zero = (theta / (theta + mu)) ** theta
    print(name, "observed zero fraction:", round(float(np.mean(X_test == 0)), 4),
          "predicted zero probability:", round(float(p_zero.mean()), 4))

## Phase 5 — Save outputs and report

The models and plots have already been saved in `results/`. This final cell saves your selected gene names and creates a concise results summary you can edit for your report.

**Next research steps:** add external cell-type annotations for a linear probe, evaluate across multiple seeds/latent dimensions, inspect gene-wise variance, and test donor/batch generalization with suitable datasets. No claim of publication-level novelty is made.

In [ ]:
np.save("results/selected_genes.npy", genes.to_numpy())
np.save("results/gaussian_test_latent.npy", latent_means(gaussian_vae, X_test))
np.save("results/poisson_test_latent.npy", latent_means(poisson_vae, X_test))
np.save("results/nb_test_latent.npy", latent_means(nb_vae, X_test))

text = "# Single-cell VAE experiment summary\n\n"
text += "Data: PBMC 3k, QC filtered, train-only HVG selection (2000 genes).\n\n"
text += results.to_string(index=False) + "\n\n"
text += f"Gaussian PCA test MSE: {pca_mse:.4f}\n\n"
text += f"Gaussian VAE test MSE: {vae_mse:.4f}\n\n"
text += "Gaussian and count likelihood losses are on different scales.\n"
Path("results/summary.md").write_text(text)
print(text)
print("Results saved in /results")

## Notes for explaining the project in an interview

**Problem:** Single-cell gene expression is high-dimensional, sparse count data with varying sequencing depths.

**My approach:** Learn a 20-dimensional probabilistic representation using the same VAE encoder/decoder structure while changing the observation likelihood.

**Why three models?** Gaussian is a continuous-expression baseline, Poisson models nonnegative counts, and Negative Binomial also permits variance greater than the mean.

**What does the decoder predict?** Distribution parameters (continuous mean, Poisson rate, or NB mean/dispersion), **not** a cell-type label or an exact deterministic gene count.

**What do we conclude?** Only after running all experiments and comparing held-out statistics. Better-looking UMAP clusters alone are not sufficient evidence.